# Targeted amplification analysis

Run the setup cell first. Review model, SAE, data paths, and run settings before starting. See the repository README for setup and pipeline order.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os

CACHE_ROOT = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"
os.environ["TRANSFORMERS_CACHE"] = f"{CACHE_ROOT}/transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{CACHE_ROOT}/hub"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Authentication is read from HF_TOKEN or the local Hugging Face login.
# Authentication is read from HF_TOKEN or the local Hugging Face login.

In [ ]:
import gc
from typing import List, Tuple, Dict, Optional

import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report

from sae_emotion.logit_ablation_utils import (
    load_llm,
    build_emotion_token_map,
    SAECache,
    zero_shot_template,
    cleanup_memory,
)


# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]
FEATURE_SETS_BY_MODEL = {
    "google/gemma-2-2b": [
        [(22, 11640)],
        [(22, 15295)],
    ],
    "google/gemma-2-9b": [
        [(35, 14786), (35, 10444)],
        [(35, 15406)],
    ],
    "google/gemma-3-4b-pt": [
        [(31, 1519)],
        [(31, 13001), (31, 1195)],
    ],
    "google/gemma-3-4b-it": [
        [(31, 2247), (31, 13175), (31, 2247), (31, 799)],
        [(31, 13001)],
    ],
    "google/gemma-3-12b-pt": [
        [(42, 12673), (42, 10933), (42, 8678)],
        [(42, 3753)],
    ],
    "google/gemma-3-12b-it": [
        [(42, 8678)],
        [(42, 3753)],
    ],
    "meta-llama/Llama-3.1-8B": [
        [(26, 4352)],
        [(26, 11515)],
    ],
}

MULTIPLIER_VALUES = [1.2, 1.5, 2, 4, 8, 12, 16]

DATASET_CSV_PATH = "data/val_df.csv"

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

DEVICE = "cuda"
DTYPE = torch.bfloat16
CACHE_DIR = None

TEXT_COL = "input_text"
LABEL_COL = "emotion"

BATCH_SIZE = 128
APPLY_PROMPT_TEMPLATE = False
MAX_EXAMPLES = None


# ============================================================
# Global runtime objects
# These are reset for each model in the model loop
# ============================================================

llm = None
emotion_to_token = None
emotion_token_ids = None
sae_cache = None


# ============================================================
# Helpers
# ============================================================

def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def get_sae_size_for_model(model_name: str):
    if "gemma-3" in model_name:
        return "small"
    return "big"

def normalize_feature_set(target_features):
    if isinstance(target_features, tuple):
        return [target_features]
    return list(target_features)


def feature_set_tag(target_features) -> str:
    target_features = normalize_feature_set(target_features)
    return "__".join([f"L{l}_F{f}" for l, f in target_features])


def build_steering_dict(
    target_features: List[Tuple[int, int]],
    multipliers: List[float],
) -> Dict[int, Dict[int, float]]:
    if len(target_features) != len(multipliers):
        raise ValueError("target_features and multipliers must have the same length.")

    steering_dict = {}

    for (layer_idx, feature_idx), multiplier in zip(target_features, multipliers):
        layer_idx = int(layer_idx)
        feature_idx = int(feature_idx)
        multiplier = float(multiplier)

        if layer_idx not in steering_dict:
            steering_dict[layer_idx] = {}

        steering_dict[layer_idx][feature_idx] = multiplier

    return steering_dict


def multiply_sae_features(
    sae,
    source_sae_features: torch.Tensor,
    feature_to_multiplier: Dict[int, float],
    residual: torch.Tensor,
) -> torch.Tensor:
    modified_sae_features = source_sae_features.clone()

    for feature_idx, multiplier in feature_to_multiplier.items():
        feature_idx = int(feature_idx)
        multiplier = float(multiplier)

        modified_sae_features[:, feature_idx] = (
            modified_sae_features[:, feature_idx] * multiplier
        )

    modified_hidden = sae.decode(modified_sae_features)
    return modified_hidden + residual


# ============================================================
# Prediction with optional steering
# ============================================================

@torch.no_grad()
def predict_batch_with_optional_steering(
    examples: List[str],
    steering_dict: Optional[Dict[int, Dict[int, float]]] = None,
) -> List[str]:
    global llm, emotion_token_ids, sae_cache

    if steering_dict is None or len(steering_dict) == 0:
        with llm.trace(examples):
            emotion_logits = llm.output.logits[:, -1, emotion_token_ids].save()

        pred_idx = emotion_logits.argmax(dim=-1).detach().cpu().tolist()
        return [EMOTIONS[i] for i in pred_idx]

    layer_cache = {}

    with llm.trace(examples):
        for layer_idx in sorted(steering_dict.keys()):
            layer_idx = int(layer_idx)

            source_hidden = (
                llm.model.layers[layer_idx]
                .output[:, -1, :]
                .save()
            )

            layer_cache[layer_idx] = source_hidden

    steered_hidden_by_layer = {}

    for layer_idx, source_hidden in layer_cache.items():
        sae = sae_cache.load(layer_idx)

        source_sae_features = sae.encode(source_hidden)
        source_hidden_decoded = sae.decode(source_sae_features)
        residual = source_hidden - source_hidden_decoded

        steered_hidden = multiply_sae_features(
            sae=sae,
            source_sae_features=source_sae_features,
            feature_to_multiplier=steering_dict[layer_idx],
            residual=residual,
        )

        steered_hidden_by_layer[layer_idx] = steered_hidden

    with llm.trace(examples):
        for layer_idx, steered_hidden in steered_hidden_by_layer.items():
            layer_idx = int(layer_idx)

            llm.model.layers[layer_idx].output[:, -1, :] = steered_hidden

        emotion_logits = llm.output.logits[:, -1, emotion_token_ids].save()

    pred_idx = emotion_logits.argmax(dim=-1).detach().cpu().tolist()
    pred_emotions = [EMOTIONS[i] for i in pred_idx]

    cleanup_memory()

    return pred_emotions


# ============================================================
# Evaluation
# ============================================================

def evaluate_before_after_steering(
    dataset_csv_path: str,
    steering_dict: Dict[int, Dict[int, float]],
    batch_size: int = 128,
    text_col: str = "input_text",
    label_col: str = "emotion",
    apply_prompt_template: bool = False,
    max_examples: Optional[int] = None,
):
    df = pd.read_csv(dataset_csv_path)

    df = df[[text_col, label_col]].dropna().copy()
    df[label_col] = df[label_col].astype(str).str.lower().str.strip()
    df = df[df[label_col].isin(EMOTIONS)].copy()

    if max_examples is not None:
        df = df.iloc[:max_examples].copy()

    if apply_prompt_template:
        df["_prompt"] = df[text_col].apply(zero_shot_template)
    else:
        df["_prompt"] = df[text_col].astype(str)

    preds_before = []
    preds_after = []

    for start in tqdm(range(0, len(df), batch_size), desc="Evaluating"):
        batch_df = df.iloc[start:start + batch_size]
        examples = batch_df["_prompt"].tolist()

        batch_preds_before = predict_batch_with_optional_steering(
            examples=examples,
            steering_dict=None,
        )

        batch_preds_after = predict_batch_with_optional_steering(
            examples=examples,
            steering_dict=steering_dict,
        )

        preds_before.extend(batch_preds_before)
        preds_after.extend(batch_preds_after)

    df["pred_before"] = preds_before
    df["pred_after"] = preds_after

    df["correct_before"] = df["pred_before"] == df[label_col]
    df["correct_after"] = df["pred_after"] == df[label_col]
    df["changed"] = df["pred_before"] != df["pred_after"]

    y_true = df[label_col].tolist()
    y_before = df["pred_before"].tolist()
    y_after = df["pred_after"].tolist()

    before_macro_f1 = f1_score(
        y_true, y_before, labels=EMOTIONS, average="macro", zero_division=0
    )
    after_macro_f1 = f1_score(
        y_true, y_after, labels=EMOTIONS, average="macro", zero_division=0
    )

    before_weighted_f1 = f1_score(
        y_true, y_before, labels=EMOTIONS, average="weighted", zero_division=0
    )
    after_weighted_f1 = f1_score(
        y_true, y_after, labels=EMOTIONS, average="weighted", zero_division=0
    )

    before_acc = df["correct_before"].mean()
    after_acc = df["correct_after"].mean()

    summary = {
        "before_accuracy": float(before_acc),
        "after_accuracy": float(after_acc),
        "delta_accuracy": float(after_acc - before_acc),
        "before_macro_f1": float(before_macro_f1),
        "after_macro_f1": float(after_macro_f1),
        "delta_macro_f1": float(after_macro_f1 - before_macro_f1),
        "before_weighted_f1": float(before_weighted_f1),
        "after_weighted_f1": float(after_weighted_f1),
        "delta_weighted_f1": float(after_weighted_f1 - before_weighted_f1),
        "n_examples": int(len(df)),
        "n_changed_predictions": int(df["changed"].sum()),
        "changed_prediction_rate": float(df["changed"].mean()),
    }

    before_report = classification_report(
        y_true,
        y_before,
        labels=EMOTIONS,
        output_dict=True,
        zero_division=0,
    )

    after_report = classification_report(
        y_true,
        y_after,
        labels=EMOTIONS,
        output_dict=True,
        zero_division=0,
    )

    per_emotion_rows = []

    for emotion in EMOTIONS:
        emotion_mask = df[label_col] == emotion

        row = {
            "emotion": emotion,
            "n": int(emotion_mask.sum()),

            "precision_before": float(before_report[emotion]["precision"]),
            "recall_before": float(before_report[emotion]["recall"]),
            "f1_before": float(before_report[emotion]["f1-score"]),

            "precision_after": float(after_report[emotion]["precision"]),
            "recall_after": float(after_report[emotion]["recall"]),
            "f1_after": float(after_report[emotion]["f1-score"]),

            "delta_precision": float(
                after_report[emotion]["precision"]
                - before_report[emotion]["precision"]
            ),
            "delta_recall": float(
                after_report[emotion]["recall"]
                - before_report[emotion]["recall"]
            ),
            "delta_f1": float(
                after_report[emotion]["f1-score"]
                - before_report[emotion]["f1-score"]
            ),

            "change_rate": float(df.loc[emotion_mask, "changed"].mean()),
        }

        per_emotion_rows.append(row)

    per_emotion = pd.DataFrame(per_emotion_rows)

    print("\nOverall performance")
    print(pd.DataFrame([summary]))

    print("\nPer-emotion F1 summary")
    print(per_emotion)

    return summary, per_emotion, df


# ============================================================
# Multiplier sweep
# ============================================================

def run_multiplier_sweep(
    model_name: str,
    sae_size,
    target_features,
    multiplier_values: List[float],
    dataset_csv_path: str,
    output_csv_path: str,
    batch_size: int = 128,
    text_col: str = "input_text",
    label_col: str = "emotion",
    apply_prompt_template: bool = False,
    max_examples: Optional[int] = None,
):
    target_features = normalize_feature_set(target_features)

    output_dir = os.path.dirname(output_csv_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    per_emotion_path = output_csv_path.replace(".csv", "_per_emotion.csv")
    all_predictions_path = output_csv_path.replace(".csv", "_all_predictions.csv")

    rows = []
    per_emotion_dfs = []
    all_pred_dfs = []

    for multiplier in multiplier_values:
        print("\n" + "=" * 80)
        print(f"Model: {model_name}")
        print(f"SAE size: {sae_size}")
        print(f"Target features: {target_features}")
        print(f"Multiplier: {multiplier}")
        print("=" * 80)

        steering_dict = build_steering_dict(
            target_features=target_features,
            multipliers=[multiplier] * len(target_features),
        )

        summary, per_emotion, pred_df = evaluate_before_after_steering(
            dataset_csv_path=dataset_csv_path,
            steering_dict=steering_dict,
            batch_size=batch_size,
            text_col=text_col,
            label_col=label_col,
            apply_prompt_template=apply_prompt_template,
            max_examples=max_examples,
        )

        row = {
            "model_name": model_name,
            "sae_size": sae_size,
            "feature_set": feature_set_tag(target_features),
            "features": str(target_features),
            "multiplier": float(multiplier),
            **summary,
        }

        rows.append(row)

        per_emotion_copy = per_emotion.copy()
        per_emotion_copy["model_name"] = model_name
        per_emotion_copy["sae_size"] = sae_size
        per_emotion_copy["feature_set"] = feature_set_tag(target_features)
        per_emotion_copy["features"] = str(target_features)
        per_emotion_copy["multiplier"] = float(multiplier)
        per_emotion_dfs.append(per_emotion_copy)

        pred_df_copy = pred_df.copy()
        pred_df_copy["model_name"] = model_name
        pred_df_copy["sae_size"] = sae_size
        pred_df_copy["feature_set"] = feature_set_tag(target_features)
        pred_df_copy["features"] = str(target_features)
        pred_df_copy["multiplier"] = float(multiplier)
        all_pred_dfs.append(pred_df_copy)

        pd.DataFrame(rows).to_csv(output_csv_path, index=False)

        pd.concat(per_emotion_dfs, ignore_index=True).to_csv(
            per_emotion_path,
            index=False,
        )

        pd.concat(all_pred_dfs, ignore_index=True).to_csv(
            all_predictions_path,
            index=False,
        )

        print(f"Saved summary to: {output_csv_path}")
        print(f"Saved per-emotion results to: {per_emotion_path}")
        print(f"Saved all predictions to: {all_predictions_path}")

    sweep_df = pd.DataFrame(rows)
    per_emotion_df = pd.concat(per_emotion_dfs, ignore_index=True)
    all_predictions_df = pd.concat(all_pred_dfs, ignore_index=True)

    return sweep_df, per_emotion_df, all_predictions_df


# ============================================================
# Run all models
# ============================================================

all_sweep_dfs = []
all_per_emotion_dfs = []

for MODEL_NAME in MODELS:
    print("\n" + "#" * 100)
    print(f"Loading model: {MODEL_NAME}")
    print("#" * 100)

    SAE_SIZE = get_sae_size_for_model(MODEL_NAME)
    model_slug = model_safe_name(MODEL_NAME)

    llm = load_llm(
        model_name=MODEL_NAME,
        device=DEVICE,
        dtype=DTYPE,
        cache_dir=CACHE_DIR,
    )

    emotion_to_token = build_emotion_token_map(
        llm=llm,
        emotions=EMOTIONS,
    )

    emotion_token_ids = [emotion_to_token[e] for e in EMOTIONS]

    sae_cache = SAECache(
        model_name=MODEL_NAME,
        device=DEVICE,
        sae_size=SAE_SIZE,
    )

    for run_idx, target_features in enumerate(FEATURE_SETS_BY_MODEL[MODEL_NAME], start=1):
        tag = feature_set_tag(target_features)

        output_dir = f"data/{model_slug}-feature-act/manual_weak_emotion_boosting"
        os.makedirs(output_dir, exist_ok=True)

        sweep_output_path = os.path.join(
            output_dir,
            f"multiplier_sweep_run{run_idx}_{tag}.csv",
        )

        sweep_df, per_emotion_df, all_predictions_df = run_multiplier_sweep(
            model_name=MODEL_NAME,
            sae_size=SAE_SIZE,
            target_features=target_features,
            multiplier_values=MULTIPLIER_VALUES,
            dataset_csv_path=DATASET_CSV_PATH,
            output_csv_path=sweep_output_path,
            batch_size=BATCH_SIZE,
            text_col=TEXT_COL,
            label_col=LABEL_COL,
            apply_prompt_template=APPLY_PROMPT_TEMPLATE,
            max_examples=MAX_EXAMPLES,
        )

        sweep_df["run_idx"] = run_idx
        per_emotion_df["run_idx"] = run_idx

        all_sweep_dfs.append(sweep_df)
        all_per_emotion_dfs.append(per_emotion_df)

    del llm
    del sae_cache
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Save aggregate files
# ============================================================

all_sweeps_df = pd.concat(all_sweep_dfs, ignore_index=True)
all_per_emotion_df = pd.concat(all_per_emotion_dfs, ignore_index=True)

all_sweeps_df.to_csv(
    "data/all_models_manual_weak_emotion_boosting_summary.csv",
    index=False,
)

all_per_emotion_df.to_csv(
    "data/all_models_manual_weak_emotion_boosting_per_emotion.csv",
    index=False,
)

print("\nDone.")
print("Saved overall summary to:")
print("data/all_models_manual_weak_emotion_boosting_summary.csv")
print("Saved per-emotion summary to:")
print("data/all_models_manual_weak_emotion_boosting_per_emotion.csv")

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

BASE_DIRS = [
    "data/google-gemma-2-2b-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-2-9b-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-4b-pt-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-4b-it-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-12b-pt-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-12b-it-feature-act/manual_weak_emotion_boosting",
]

OUTPUT_DIR = "paper_visuals/manual_weak_emotion_boosting"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

MODEL_ORDER = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

MODEL_LABELS = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
}

MODEL_FROM_DIR = {
    "google-gemma-2-2b-feature-act": "google/gemma-2-2b",
    "google-gemma-2-9b-feature-act": "google/gemma-2-9b",
    "google-gemma-3-4b-pt-feature-act": "google/gemma-3-4b-pt",
    "google-gemma-3-4b-it-feature-act": "google/gemma-3-4b-it",
    "google-gemma-3-12b-pt-feature-act": "google/gemma-3-12b-pt",
    "google-gemma-3-12b-it-feature-act": "google/gemma-3-12b-it",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Loading
# ============================================================

def infer_run_idx(path):
    name = os.path.basename(path)
    if "run1" in name:
        return 1
    if "run2" in name:
        return 2
    return None


def infer_model_name(path):
    parts = path.split(os.sep)
    for part in parts:
        if part in MODEL_FROM_DIR:
            return MODEL_FROM_DIR[part]
    return None


def load_all_results_from_model_folders():
    summary_dfs = []
    per_emotion_dfs = []

    for base_dir in BASE_DIRS:
        summary_paths = sorted(
            glob.glob(os.path.join(base_dir, "multiplier_sweep_run*.csv"))
        )

        summary_paths = [
            p for p in summary_paths
            if not p.endswith("_per_emotion.csv")
            and not p.endswith("_all_predictions.csv")
        ]

        per_emotion_paths = sorted(
            glob.glob(os.path.join(base_dir, "multiplier_sweep_run*_per_emotion.csv"))
        )

        for path in summary_paths:
            df = pd.read_csv(path)
            df["run_idx"] = infer_run_idx(path)
            df["model_name"] = infer_model_name(path)
            df["source_file"] = path
            summary_dfs.append(df)

        for path in per_emotion_paths:
            df = pd.read_csv(path)
            df["run_idx"] = infer_run_idx(path)
            df["model_name"] = infer_model_name(path)
            df["source_file"] = path
            per_emotion_dfs.append(df)

    if not summary_dfs:
        raise FileNotFoundError("No summary files found.")

    if not per_emotion_dfs:
        raise FileNotFoundError("No per-emotion files found.")

    summary_df = pd.concat(summary_dfs, ignore_index=True)
    per_emotion_df = pd.concat(per_emotion_dfs, ignore_index=True)

    summary_df = summary_df.dropna(subset=["run_idx", "model_name"])
    per_emotion_df = per_emotion_df.dropna(subset=["run_idx", "model_name"])

    return summary_df, per_emotion_df


summary_df, per_emotion_df = load_all_results_from_model_folders()


# ============================================================
# Standardize columns
# ============================================================

summary_df["run_idx"] = summary_df["run_idx"].astype(int)
per_emotion_df["run_idx"] = per_emotion_df["run_idx"].astype(int)

summary_df["multiplier"] = pd.to_numeric(summary_df["multiplier"], errors="coerce")
per_emotion_df["multiplier"] = pd.to_numeric(per_emotion_df["multiplier"], errors="coerce")

summary_df["model_label"] = summary_df["model_name"].map(MODEL_LABELS)
per_emotion_df["model_label"] = per_emotion_df["model_name"].map(MODEL_LABELS)

summary_df["model_label"] = pd.Categorical(
    summary_df["model_label"],
    categories=[MODEL_LABELS[m] for m in MODEL_ORDER],
    ordered=True,
)

per_emotion_df["model_label"] = pd.Categorical(
    per_emotion_df["model_label"],
    categories=[MODEL_LABELS[m] for m in MODEL_ORDER],
    ordered=True,
)

summary_df = summary_df.sort_values(["run_idx", "model_label", "multiplier"])
per_emotion_df = per_emotion_df.sort_values(
    ["run_idx", "model_label", "emotion", "multiplier"]
)

print("Loaded summary results:")
print(summary_df[["model_name", "run_idx", "multiplier", "after_macro_f1", "changed_prediction_rate"]])

print("\nLoaded per-emotion results:")
print(per_emotion_df[["model_name", "run_idx", "emotion", "multiplier", "f1_after"]].head())


# ============================================================
# Plot helpers
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_macro_f1(summary_df, run_idx):
    df = summary_df[summary_df["run_idx"] == run_idx].copy()

    if df.empty:
        print(f"No data for run {run_idx}, macro-F1 skipped.")
        return

    plt.figure(figsize=(7.5, 4.6))

    sns.lineplot(
        data=df,
        x="multiplier",
        y="after_macro_f1",
        hue="model_label",
        hue_order=[MODEL_LABELS[m] for m in MODEL_ORDER],
        marker="o",
        linewidth=2,
        errorbar=None,
    )

    plt.xlabel("Multiplier k")
    plt.ylabel("Macro-F1")
    plt.title(f"Run {run_idx}: Macro-F1 by k")
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    savefig(os.path.join(OUTPUT_DIR, f"run{run_idx}_macro_f1_by_k.png"))


def plot_flip_rate(summary_df, run_idx):
    df = summary_df[summary_df["run_idx"] == run_idx].copy()

    if df.empty:
        print(f"No data for run {run_idx}, flip-rate skipped.")
        return

    plt.figure(figsize=(7.5, 4.6))

    sns.lineplot(
        data=df,
        x="multiplier",
        y="changed_prediction_rate",
        hue="model_label",
        hue_order=[MODEL_LABELS[m] for m in MODEL_ORDER],
        marker="o",
        linewidth=2,
        errorbar=None,
    )

    plt.xlabel("Multiplier k")
    plt.ylabel("Flip rate")
    plt.title(f"Run {run_idx}: Flip rate by k")
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    savefig(os.path.join(OUTPUT_DIR, f"run{run_idx}_flip_rate_by_k.png"))


def plot_per_emotion_f1_for_each_model(per_emotion_df, run_idx):
    df_run = per_emotion_df[per_emotion_df["run_idx"] == run_idx].copy()

    if df_run.empty:
        print(f"No data for run {run_idx}, per-emotion F1 skipped.")
        return

    for model_name in MODEL_ORDER:
        model_label = MODEL_LABELS[model_name]
        df_model = df_run[df_run["model_name"] == model_name].copy()

        if df_model.empty:
            print(f"No data for run {run_idx}, {model_label}, skipped.")
            continue

        plt.figure(figsize=(6.8, 4.5))

        sns.lineplot(
            data=df_model,
            x="multiplier",
            y="f1_after",
            hue="emotion",
            hue_order=EMOTIONS,
            marker="o",
            linewidth=2,
            errorbar=None,
        )

        plt.xlabel("Multiplier k")
        plt.ylabel("Per-emotion F1")
        plt.title(f"Run {run_idx}: {model_label}")
        plt.legend(title="Emotion", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

        safe_model = model_name.replace("/", "-")
        savefig(
            os.path.join(
                OUTPUT_DIR,
                f"run{run_idx}_{safe_model}_per_emotion_f1_by_k.png",
            )
        )


# ============================================================
# Run plots
# ============================================================

for run_idx in [1, 2]:
    plot_macro_f1(summary_df, run_idx)
    plot_flip_rate(summary_df, run_idx)
    plot_per_emotion_f1_for_each_model(per_emotion_df, run_idx)

print(f"\nSaved plots to: {OUTPUT_DIR}")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Assumes these already exist from your previous full code:
# - per_emotion_df
# - EMOTIONS
# - OUTPUT_DIR
# - savefig

# ============================================================
# Aggregate plots across models with shaded variance area
# Each line = emotion
# Shaded area = mean ± std across models
# ============================================================

def aggregate_mean_std(df, value_col):
    agg = (
        df.groupby(["run_idx", "emotion", "multiplier"], as_index=False)
        .agg(
            mean_value=(value_col, "mean"),
            std_value=(value_col, "std"),
            n_models=("model_name", "nunique"),
        )
    )

    agg["std_value"] = agg["std_value"].fillna(0.0)
    return agg


def plot_aggregate_per_emotion_f1(per_emotion_df, run_idx):
    df = per_emotion_df[per_emotion_df["run_idx"] == run_idx].copy()
    agg = aggregate_mean_std(df, value_col="f1_after")

    plt.figure(figsize=(7.2, 4.8))

    for emotion in EMOTIONS:
        sub = agg[agg["emotion"] == emotion].sort_values("multiplier")

        x = sub["multiplier"].to_numpy()
        mean = sub["mean_value"].to_numpy()
        std = sub["std_value"].to_numpy()

        plt.plot(x, mean, marker="o", linewidth=2, label=emotion)
        plt.fill_between(x, mean - std, mean + std, alpha=0.18)

    plt.xlabel("Multiplier k")
    plt.ylabel("Mean per-emotion F1")
    plt.title(f"Run {run_idx}: Aggregate per-emotion F1 across models")
    plt.legend(title="Emotion", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    savefig(
        os.path.join(
            OUTPUT_DIR,
            f"run{run_idx}_aggregate_per_emotion_f1_across_models_shaded.png",
        )
    )


def plot_aggregate_per_emotion_flip_rate(per_emotion_df, run_idx):
    df = per_emotion_df[per_emotion_df["run_idx"] == run_idx].copy()
    agg = aggregate_mean_std(df, value_col="change_rate")

    plt.figure(figsize=(7.2, 4.8))

    for emotion in EMOTIONS:
        sub = agg[agg["emotion"] == emotion].sort_values("multiplier")

        x = sub["multiplier"].to_numpy()
        mean = sub["mean_value"].to_numpy()
        std = sub["std_value"].to_numpy()

        plt.plot(x, mean, marker="o", linewidth=2, label=emotion)
        plt.fill_between(x, mean - std, mean + std, alpha=0.18)

    plt.xlabel("Multiplier k")
    plt.ylabel("Mean flip rate")
    plt.title(f"Run {run_idx}: Aggregate per-emotion flip rate across models")
    plt.legend(title="Emotion", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    savefig(
        os.path.join(
            OUTPUT_DIR,
            f"run{run_idx}_aggregate_per_emotion_flip_rate_across_models_shaded.png",
        )
    )


# ============================================================
# Run aggregate plots
# ============================================================

for run_idx in [1, 2]:
    plot_aggregate_per_emotion_f1(per_emotion_df, run_idx)
    plot_aggregate_per_emotion_flip_rate(per_emotion_df, run_idx)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Assumes these already exist:
# - per_emotion_df
# - EMOTIONS
# - OUTPUT_DIR
# - savefig

RUN_LABELS = {
    1: "Enhancing Disgust Features",
    2: "Enhancing Surprise Features",
}

def aggregate_mean_std(df, value_col):
    agg = (
        df.groupby(["run_idx", "emotion", "multiplier"], as_index=False)
        .agg(
            mean_value=(value_col, "mean"),
            std_value=(value_col, "std"),
            n_models=("model_name", "nunique"),
        )
    )

    agg["std_value"] = agg["std_value"].fillna(0.0)
    return agg


def plot_aggregate_per_emotion_f1_two_runs(per_emotion_df):
    agg = aggregate_mean_std(per_emotion_df, value_col="f1_after")

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(5, 3),
        sharey=True,
    )

    for ax, run_idx in zip(axes, [1, 2]):
        df_run = agg[agg["run_idx"] == run_idx].copy()

        for emotion in EMOTIONS:
            sub = df_run[df_run["emotion"] == emotion].sort_values("multiplier")

            if sub.empty:
                continue

            x = sub["multiplier"].to_numpy()
            mean = sub["mean_value"].to_numpy()
            std = sub["std_value"].to_numpy()

            lower = np.clip(mean - std, 0, 1)
            upper = np.clip(mean + std, 0, 1)

            ax.plot(
                x,
                mean,
                marker="o",
                linewidth=2,
                label=emotion,
            )

            ax.fill_between(
                x,
                lower,
                upper,
                alpha=0.16,
            )

        ax.set_title(RUN_LABELS[run_idx])
        ax.set_xlabel(r"Multiplier $\alpha$")
        ax.set_ylim(0, 1.02)
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Mean per-emotion F1")

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        title="Emotion",
        loc="center left",
        bbox_to_anchor=(0.98, 0.7),
        frameon=False,
    )

    savefig(
        os.path.join(
            OUTPUT_DIR,
            "figure3.png",
        )
    )

plot_aggregate_per_emotion_f1_two_runs(per_emotion_df)